# Plot g_mem


In [5]:
import sys
# Path to nrv (if dev version conda env)
sys.path.append("../../")
import nrv
import numpy as np
import matplotlib.pyplot as plt



# Axon Simulations

- Myelinated axon:

In [6]:
Nseg=3
y = 0
z = 0
d = 6
L = nrv.get_length_from_nodes(d, 33)

axonm = nrv.myelinated(y,z,d,L,dt=0.005,rec='all', Nseg_per_sec=Nseg-1,model='MRG')

t_start = 1
duration = 0.1
amplitude = 2
position = .5
axonm.insert_I_Clamp(position, t_start, duration, amplitude)

t_sim=5
resultsm = axonm.simulate(t_sim=t_sim, record_V_mem=False, record_g_mem=True,record_g_ions=True)

del axonm


- Unmyelinated axon

In [7]:
Nseg=2
y = 0
z = 0
d = 1
L = 2300

axonu = nrv.unmyelinated(y,z,d,L,dt=0.005,Nsec=3000)
print(axonu.Nseg)

t_start = 1
duration = 0.1
amplitude = 2
position = .5
axonu.insert_I_Clamp(position, t_start, duration, amplitude)

t_sim=5
resultsu = axonu.simulate(t_sim=t_sim, record_V_mem=False, record_g_mem=True)

del axonu

3000


# Animations

In [ ]:
import matplotlib.animation as animation

xu = resultsu["x_rec"]
gu = resultsu["g_mem"]      # shape (n_x, n_t)
tu = resultsu["t"]

xm = resultsm["x_rec"]
gm = resultsm["g_mem"]      # shape (n_x, n_t)
tm = resultsm["t"]
dt = float(tm[1] - tm[0])



# downsample frames if there are many timepoints
max_frames = 200
step = max(1, len(tm) // max_frames)
frame_indices = list(range(0, len(tm), step))
interval_ms = dt * 1000 * step  # ms between frames
fps = 500.0 / interval_ms

fig, axs = plt.subplots(2, figsize=(10, 5), layout="constrained")
lineu, = axs[0].plot(xu, gu[:, frame_indices[0]], color="C0")
axs[0].set_xlim(xu.min(), xu.max())
axs[0].set_ylim(np.nanmin(gu), np.nanmax(gu))
axs[0].set_xlabel("x-position")
axs[0].set_ylabel("g_mem")
title = axs[0].set_title(f"Unmyelinated: time = {tu[frame_indices[0]]:.3f} s")

linem, = axs[1].plot(xm, gm[:, frame_indices[0]], color="C0")
axs[1].set_xlim(xm.min(), xm.max())
axs[1].set_ylim(np.nanmin(gm), np.nanmax(gm))
axs[1].set_xlabel("x-position")
axs[1].set_ylabel("g_mem")
titlem = axs[1].set_title(f"Myelinated: time = {tu[frame_indices[0]]:.3f} s")


def update_frame(frame_idx):
    i = frame_indices[frame_idx]
    lineu.set_ydata(gu[:, i])
    linem.set_ydata(gm[:, i])
    title.set_text(f"Unmyelinated: time = {tu[i]:.3f} s")
    titlem.set_text(f"Myelinated: time = {tm[i]:.3f} s")
    return (lineu, title)


ani = animation.FuncAnimation(
    fig,
    update_frame,
    frames=len(frame_indices),
    interval=interval_ms,
    blit=True,
)

# Save animation: try mp4 (ffmpeg) first, fallback to gif (Pillow)
# output_mp4 = "g_mem.mp4"
# output_gif = "g_mem_middle_stim.gif"
# try:
#     writer = animation.FFMpegWriter(fps=fps)
#     ani.save(output_mp4, writer=writer, dpi=150)
#     print(f"Saved animation to {output_mp4}")
# except Exception:
#     writer = animation.PillowWriter(fps=fps)
#     ani.save(output_gif, writer=writer)
#     print(f"Saved animation to {output_gif}")

plt.close(fig)

Saved animation to g_mem_middle_stim.gif
